In [61]:
import math
import random
import tqdm

The following code contains a basic definition of the `Value` object which wraps a value and adds support for autograd and backpropagation for several operators:
- `+`
- `-` (by using `+ (-x)`)
- `*`
- `/` (by using `**-1`)
- `**` (pow)
- `exp`
- `tanh` (used as activation function)

In [ ]:
class Value:
    def __init__(self, data, _prev = ()):
        self.data = data
        self.grad = 0.0
        self._prev = set(_prev)
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
 
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out

    def __radd__(self, other):
        return self + other

    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __rmul__(self, other):
        return self * other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other):
        return -(self - other)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "Unsupported exponent type (only accept int/float)."
        out = Value(self.data ** other, (self, ))

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * (other**-1)

    def __rtruediv__(self, other):
        return other * (self**-1)

    def exp(self):
        out =  Value(math.exp(self.data), (self, ))
        
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        t = (math.exp(2 * self.data) - 1) / (math.exp(2 * self.data) + 1)
        out = Value(t, (self, ))

        def _backward():
            self.grad = (1 - t**2) * out.grad
        out._backward = _backward

        return out

    def __repr__(self):
        return f"Value(data={self.data})"

    
    def backward(self):
        topo: list[Value] = []
        vis: set[Value] = set()

        def build(v: Value):
            if v not in vis:
                vis.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

In [3]:
a = Value(2.0); b = Value(3.0)
c = a*b + a/b - (a**2) + b.exp()
c.backward()
print(a.grad, b.grad)

-0.6666666666666667 21.863314700965446


The following code contains the definitions of a single neuron, a layer and an MLP (multi-layer perceptron), each of which receive a vector of `nin` variables and gives and output of `nouts[-1]` (which is the last element of `nouts`).

In [ ]:
class Neuron:
    def __init__(self, nin): # nin means the number of variables of the input vector
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]    

class Layer:
    def __init__(self, nin, nout): # nout means the number of neurons in the layer
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self, nin, nouts): # nouts is a list containing each layer's size
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for neuron in layer.neurons for p in neuron.parameters()]

In [45]:
x = [2.0, 3.0]
n = MLP(2, [3, 3, 1])
n(x)
n.parameters()


[Value(data=-0.6300561172051233),
 Value(data=-0.7522096711511366),
 Value(data=-0.5788469802267071),
 Value(data=0.6014931807083619),
 Value(data=0.8739383172891615),
 Value(data=-0.9544348486626832),
 Value(data=-0.14876233606636569),
 Value(data=-0.7969995612516605),
 Value(data=-0.48016022041433604),
 Value(data=-0.5583414573673653),
 Value(data=0.2938514396706451),
 Value(data=-0.2994120652069354),
 Value(data=-0.6393641969406243),
 Value(data=0.007273010419774462),
 Value(data=-0.9212425858306152),
 Value(data=-0.7981575176220668),
 Value(data=0.9764702974450021),
 Value(data=-0.601288419065874),
 Value(data=-0.2828893973767963),
 Value(data=0.4631966124507212),
 Value(data=0.6766531303868326),
 Value(data=0.8369641239906629),
 Value(data=-0.6611507878050646),
 Value(data=0.34528112714610515),
 Value(data=0.9330978060863664)]

In [54]:
# build test dataset
random.seed(0) # for making fixed dataset, convenient for debug
xs = [[random.uniform(-5, 5) for _ in range(3)] for _ in range(5)]
ys = [random.uniform(-1, 1) for _ in range(5)]
xs, ys

([[3.4442185152504816, 2.5795440294030243, -0.79428419169155],
  [-2.4108324970703663, 0.11274721368608542, -0.9506586254958567],
  [2.837985890347726, -1.9668727392107255, -0.23403045847644144],
  [0.8338203945503118, 4.081128851953352, 0.046868558173902564],
  [-2.1816215560029617, 2.5580420415722394, 1.1836899667533167]],
 [-0.4989873172751189,
  0.8194925119364802,
  0.9655709520753062,
  0.6204344719931791,
  0.8043319008791654])

In [55]:
n = MLP(3, [4, 4, 1])
y_pred = [n(x) for x in xs]
y_pred

[Value(data=0.7122572514070812),
 Value(data=0.941898455252392),
 Value(data=0.8788804300370227),
 Value(data=0.28874903517730593),
 Value(data=0.3812561346557399)]

In [57]:
loss = sum((yi - yi_pred) ** 2 for yi, yi_pred in zip(ys, y_pred))
loss.backward()
loss, n.layers[0].neurons[0].w[0].grad

(Value(data=1.778620199693733), -23.06306695644196)

In [66]:
# gradient descent

lr = -1e-2
while loss.data >= 1e-4:
    y_pred = [n(x) for x in xs]
    loss = sum((yi - yi_pred) ** 2 for yi, yi_pred in zip(ys, y_pred))

    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    for p in n.parameters():
        p.data += lr * p.grad

print(loss)
y_pred = [n(x) for x in xs]
delta_y = [yi - yi_pred for yi, yi_pred in zip(ys, y_pred)]
delta_y
    

Value(data=9.985921204331594e-05)


[Value(data=-0.00018655011198565052),
 Value(data=-0.00045811734259593173),
 Value(data=0.009970777735485381),
 Value(data=0.00022080971512739023),
 Value(data=5.4381179007689795e-05)]